# Table 4 (protein): Pooled Significance Test vs. LogReg Baseline

Table 4 (`tab:perc_discovered_combined`) reports single point-estimate Precision@10/Recall@10 values
per drug/model (not averaged across CV folds, per its own caption), so a per-drug/per-fold Wilcoxon
test (as used for Table 16/Task 1) is not possible here. Instead, following the same across-drug
pooled approach used for Table 17 and the Task 2 lineage-delta significance test, we run one paired
Wilcoxon signed-rank test per model (vs. LogReg), pairing each drug's Precision@10 (and, separately,
Recall@10) across all 10 drugs. Holm-Bonferroni correction is applied within each metric's family of
4 models.

In [1]:
import pandas as pd, numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

df = pd.read_csv("data/latest/results/interpretability/combined/combined_precision_recall_all_models_druglevel.csv")
df10 = df[df["k_req"] == 10]

BASELINE = "logreg"
MODELS = ["cnn", "full320", "pca10", "transformer"]
NAME_MAP = {"cnn": "CNN", "full320": "ESM-320", "pca10": "ESM-PCA10", "transformer": "Transformer"}

In [2]:
def pooled_test(metric):
    pivot = df10.pivot(index="drug", columns="variant", values=metric)
    baseline = pivot[BASELINE]
    rows = []
    for model in MODELS:
        common = sorted(set(baseline.dropna().index) & set(pivot[model].dropna().index))
        a = pivot[model].loc[common].values
        b = baseline.loc[common].values
        _, p = wilcoxon(a, b, alternative="two-sided")
        rows.append({
            "model": NAME_MAP[model],
            f"mean_logreg_{metric}": b.mean(),
            f"mean_model_{metric}": a.mean(),
            f"delta_{metric}": a.mean() - b.mean(),
            f"p_{metric}": p,
            "n_drugs": len(common),
        })
    res = pd.DataFrame(rows).set_index("model")
    res[f"p_{metric}_holm"] = multipletests(res[f"p_{metric}"], method="holm")[1]
    return res

precision_res = pooled_test("precision")
recall_res = pooled_test("recall")
result = precision_res.join(recall_res.drop(columns=["n_drugs"]))
result.to_csv("data/latest/results/interpretability/combined/table4_pooled_significance_vs_logreg_protein.csv")
print("[OK] Wrote table4_pooled_significance_vs_logreg_protein.csv")
result.round(4)

[OK] Wrote table4_pooled_significance_vs_logreg_protein.csv


,mean_logreg_precision,mean_model_precision,delta_precision,p_precision,n_drugs,p_precision_holm,mean_logreg_recall,mean_model_recall,delta_recall,p_recall,p_recall_holm
model,,,,,,,,,,,
CNN,0.42,0.49,0.07,0.1747,10,0.3495,0.4153,0.4657,0.0503,0.4990,0.7485
ESM-320,0.42,0.32,-0.10,0.1041,10,0.3122,0.4153,0.2031,-0.2123,0.0796,0.2388
ESM-PCA10,0.42,0.40,-0.02,0.8122,10,0.8122,0.4153,0.3580,-0.0574,0.3743,0.7485
Transformer,0.42,0.26,-0.16,0.0117,10,0.0469,0.4153,0.2033,-0.2121,0.0116,0.0465
